In [1]:
import os
from tqdm import tqdm
from glob import glob

import numpy as np
import pandas as pd
import datasets
from datasets import Dataset, load_metric

from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, 
                          Seq2SeqTrainer, DataCollatorForSeq2Seq, T5Model)

import warnings
warnings.filterwarnings('ignore')

import wandb

wandb.login()

import logging
logging.basicConfig(level = logging.INFO)
transformers_logger = logging.getLogger("transformers")
transformers_logger.setLevel(logging.WARNING)

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: temuujin-razy. Use `wandb login --relogin` to force relogin


In [2]:
prefix_val = "summarize"
df_names = glob("../Lab 2/processed_dfs/*.parquet")
search_terms = '\n'.join([os.path.basename(filename).split('_')[1] for filename in df_names])

"""
df = pd.DataFrame()

for df_name in tqdm(df_names):
    df = pd.concat([df, pd.read_parquet(df_name)[['title', 'abstract']]])

df.drop_duplicates(inplace = True)
df.reset_index(drop = True, inplace = True)

df.rename(columns = {'title': 'target_text', 'abstract': 'input_text'}, inplace = True)

df['prefix'] = prefix_val

df.to_parquet("arxiv_title_generation.parquet")
"""

df = pd.read_parquet("arxiv_title_generation.parquet")

print("\nDataframe memory usage")
print(df.memory_usage(deep = True))

print(f"Dataframe shape: {df.shape}\n")
print(f"All search terms:\n{search_terms}")

print(df.head())


Dataframe memory usage
Index               132
target_text     1963896
input_text     17433653
prefix           954492
dtype: int64
Dataframe shape: (14462, 3)

All search terms:
audio+classification
audio+deep+learning
audio+encoding
audio+fast+fourier
audio+fourier
audio+generation
audio+machine+learning
audio+prediction
audio+recognition
audio+representation
audio+restoration
audio+signal
                                         target_text  \
0  Improved Mispronunciation detection system usi...   
1  Improving Factored Hybrid HMM Acoustic Modelin...   
2  Disentangling Style and Speaker Attributes for...   
3  Synthetic speech detection using meta-learning...   
4  A Pre-trained Audio-Visual Transformer for Emo...   

                                          input_text     prefix  
0  This report proposes state-of-the-art research...  summarize  
1  In this work, we show that a factored hybrid h...  summarize  
2  End-to-end neural TTS has shown improved perfo...  summarize  
3  

In [3]:
df

,target_text,input_text,prefix
0,Improved Mispronunciation detection system usi...,This report proposes state-of-the-art research...,summarize
1,Improving Factored Hybrid HMM Acoustic Modelin...,"In this work, we show that a factored hybrid h...",summarize
2,Disentangling Style and Speaker Attributes for...,End-to-end neural TTS has shown improved perfo...,summarize
3,Synthetic speech detection using meta-learning...,Recent works on speech spoofing countermeasure...,summarize
4,A Pre-trained Audio-Visual Transformer for Emo...,"In this paper, we introduce a pretrained audio...",summarize
...,...,...,...
14457,Robust Expectation-Maximization Algorithm for ...,The direction of arrival (DOA) estimation of s...,summarize
14458,Shift-Invariant Kernel Additive Modelling for ...,A major goal in blind source separation to ide...,summarize
14459,Audio style transfer,'Style transfer' among images has recently eme...,summarize
14460,Sound Source Localization in a Multipath Envir...,The propagation of sound in a shallow water en...,summarize


In [ ]:
test_size = 0.2
test_df = df.sample(frac = test_size, random_state = 1970)
train_df = df.drop(index = test_df.index)

print(f"Training instance count: {len(train_df)}\nTest instance count: {len(test_df)}\n")

train_dataset = Dataset.from_dict(train_df)
test_dataset = Dataset.from_dict(test_df)
arxiv_title_dict = datasets.DatasetDict({"train": train_dataset,"test": test_dataset})

print(arxiv_title_dict)

In [ ]:
model = T5Model(model_name = "t5-small", model_type = "t5", args = run.config, use_cuda = True)

In [ ]:
model.train_model(train_df)

In [ ]:
results = model.eval_model(test_df)

In [ ]:
def return_pred(model, test_df):
    random_num = np.random.randint(0, len(test_df))
    
    title = test_df.iloc[random_num]['target_text']
    abstract = ["summarize: " + test_df.iloc[random_num]['input_text']]
    predicted_title = model.predict(abstract)

    print(f"Test dataframe index: {random_num}")
    print(f'Actual Title: {title}')
    print(f'Predicted Title: {predicted_title[0]}')

In [ ]:
for _ in range(30):
    return_pred(model, test_df)